$$dX_t=-\nabla V(X_T)dt+BdW_t$$

$$X_t=\begin{pmatrix} x_t \\ y_t \end{pmatrix}$$

$$V(X)=\frac{1}{2}m_x^2 x^2 + \frac{1}{2}m_y^2 y^2 + m_{xy}^2xy + \frac{1}{4}\lambda_x x^4 + \frac{1}{4}\lambda_y y^4$$

$$B=\begin{pmatrix}\sigma_x & 0 \\ 0 & \sigma_y \end{pmatrix}$$

In [1]:
using Statistics
using DifferentialEquations
using Plots
using Random
using DataFrames
using Dates
using CSV

In [2]:
m2x = 1.0
m2y = 1.0
m2xy = 0.5
lambdax = 0.1
lambday = 0.1
sigmax = 0.5
sigmay = 0.5
x0 = 1.0
y0 = 0.5
tEnd = 0.5

0.5

In [3]:
A = [m2x m2xy; m2xy m2y]
B = [sigmax 0.0; 0.0 sigmay]
vecB = [sigmax, sigmay]
F = [lambdax 0.0 0.0 0.0 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0 0.0 0.0 0.0 lambday]
XIni = [x0, y0]

2-element Vector{Float64}:
 1.0
 0.5

In [4]:
function drift!(driftVec, u, p, t)
    copyto!(driftVec, A * u + F * kron(u, u, u))
end

function vol!(volVec, u, p, t)
    copyto!(volVec, vecB)
end

vol! (generic function with 1 method)

Solve original SDE

In [5]:
Random.seed!(1)
discScheme = EM()
dt = 0.0001
NPath = 1E7
NGrp = 10000
NPathPerGrp = (Int)(NPath / NGrp)
outFile = "xTyT.csv"
outDf = DataFrame([Symbol(cn) => Float64[] for cn in ["x", "y"]])

for iGrp in 1:NGrp
    sdeProb = SDEProblem(drift!, vol!, XIni, (0, tEnd))
    paths = solve(EnsembleProblem(sdeProb), discScheme, EnsembleThreads(), trajectories=NPathPerGrp, dt=dt, adaptive=false)
    for p in paths
        push!(outDf, [p.u[end][1], p.u[end][2]])
    end
    NPathGenerated = iGrp * NPathPerGrp
    CSV.write(outFile, outDf[(NPathGenerated - NPathPerGrp + 1):NPathGenerated,:], append=true)
    println(string(iGrp * NPathPerGrp) * " paths generated:" * string(now())); flush(stdout)
end

1000 paths generated:2026-09-28T18:51:56.781
2000 paths generated:2026-09-28T18:52:00.928
3000 paths generated:2026-09-28T18:52:04.884
4000 paths generated:2026-09-28T18:52:08.629
5000 paths generated:2026-09-28T18:52:12.111
6000 paths generated:2026-09-28T18:52:15.975
7000 paths generated:2026-09-28T18:52:19.490
8000 paths generated:2026-09-28T18:52:23.177
9000 paths generated:2026-09-28T18:52:27.460
10000 paths generated:2026-09-28T18:52:32.224
11000 paths generated:2026-09-28T18:52:35.900
12000 paths generated:2026-09-28T18:52:40.112
13000 paths generated:2026-09-28T18:52:43.090
14000 paths generated:2026-09-28T18:52:47.043
15000 paths generated:2026-09-28T18:52:50.595
16000 paths generated:2026-09-28T18:52:53.811
17000 paths generated:2026-09-28T18:52:58.410
18000 paths generated:2026-09-28T18:53:02.388
19000 paths generated:2026-09-28T18:53:05.843
20000 paths generated:2026-09-28T18:53:09.459
21000 paths generated:2026-09-28T18:53:13.127
22000 paths generated:2026-09-28T18:53:17.2

In [7]:
xEnds = outDf[:,1]
yEnds = outDf[:,2]
meanx = mean(xEnds)
meany = mean(yEnds)
stderrx = std(xEnds) / sqrt(NPath)
stderry = std(yEnds) / sqrt(NPath)
maxx = maximum(xEnds)
maxy = maximum(yEnds)
println(meanx, " ", stderrx, " ", maxx)
println(meany, " ", stderry, " ", maxy)

2.1736005371366933 0.00019871235960544737 8.53992588930798
1.3526649844474572 0.0001675307433920942 5.169085222865478
